# Importing Libraries and Setup

In [1]:
import os

os.environ["HF_HOME"] = f"/rs1/researchers/a/amallav/models/hf_home"
os.environ["HF_HUB_CACHE"] = os.path.join(os.environ["HF_HOME"], "hub")
os.environ["HF_HUB_OFFLINE"] = "1"   # only after cache is populated

In [2]:
import json #Used later to store the top-k predictions as a JSON string in the final CSV
from pathlib import Path
import pandas as pd
import torch
from bioclip import TreeOfLifeClassifier, Rank #TreeOfLifeClassifier = the classifier that predicts biological taxa; Rank = tells the classifier what taxonomic level you want

ModuleNotFoundError: No module named 'bioclip'

In [4]:
# os.environ["PROJECT_DIR"] = "/rs1/researchers/a/amallav/"
# print(os.environ["PROJECT_DIR"])
# os.environ["OUTPUTS_DIR"] = "/rs1/researchers/a/amallav/results"
# print(os.environ["OUTPUTS_DIR"])
# os.environ["OUTPUTS_CP_DIR"] = "/rs1/researchers/a/amallav/results/outputs_crop"
# print(os.environ["OUTPUTS_CP_DIR"])
# os.environ["OUTPUTS_BCCP_DIR"] = "/rs1/researchers/a/amallav/results/outputs_bioclip_crop"
# print(os.environ["OUTPUTS_BCCP_DIR"])

In [3]:
PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "/rs1/researchers/a/amallav/")).resolve()
OUTPUTS_DIR=Path(os.environ.get("OUTPUTS_DIR", "/rs1/researchers/a/amallav/results")).resolve()

OUTPUTS_CP_DIR = Path(os.environ.get("OUTPUTS_CP_DIR", OUTPUTS_DIR / "outputs_crop")).resolve()

OUTPUTS_BCCP_DIR = Path(os.environ.get("OUTPUTS_BCCP_DIR", OUTPUTS_DIR / "outputs_bioclip_crop")).resolve()
OUTPUTS_BCCP_DIR.mkdir(parents=True, exist_ok=True)

# CROP_OUT = PROJECT_DIR / "outputs_crop"
META_CSV = OUTPUTS_CP_DIR / "cropped_metadata.csv" #This is the metadata file that tells us what crop files exist.
OUT_CSV = OUTPUTS_BCCP_DIR / "bioclip_species_predictions.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR, "| exists:", OUTPUTS_DIR.exists())
print("OUTPUTS_CP_DIR:", OUTPUTS_CP_DIR, "| exists:", OUTPUTS_CP_DIR.exists())
print("OUTPUTS_BCCP_DIR:", OUTPUTS_BCCP_DIR, "| exists:", OUTPUTS_BCCP_DIR.exists())
print("META_CSV    :", META_CSV, "| exists:", META_CSV.exists())
print("OUT_CSV     :", OUT_CSV, "| exists:", OUT_CSV.exists())

PROJECT_DIR: /rs1/researchers/a/amallav
OUTPUTS_DIR: /rs1/researchers/a/amallav/results | exists: True
OUTPUTS_CP_DIR: /rs1/researchers/a/amallav/results/outputs_crop | exists: True
OUTPUTS_BCCP_DIR: /rs1/researchers/a/amallav/results/outputs_bioclip_crop | exists: True
META_CSV    : /rs1/researchers/a/amallav/results/outputs_crop/cropped_metadata.csv | exists: True
OUT_CSV     : /rs1/researchers/a/amallav/results/outputs_bioclip_crop/bioclip_species_predictions.csv | exists: False


# Load Bioclip 

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
# model_dir = "/share/ftrscape/{}/models/bioclip".format(__import__("os").environ["USER"])

In [5]:
MODEL_STR = "hf-hub:imageomics/bioclip-2"

TOP_K = 5 #top 5 species predictions for each image
BATCH_SIZE = 1 #This controls how many cropped images are processed at once. Might have to tune this parameter

classifier = TreeOfLifeClassifier(
    device=device,
    model_str=MODEL_STR,
)

print("Device   :", device)
print("Model    :", MODEL_STR)
print("Top-k    :", TOP_K)
print("Batch size:", BATCH_SIZE)

NameError: name 'TreeOfLifeClassifier' is not defined

# Load CSV

In [9]:
assert META_CSV.exists(), f"metadata.csv not found: {META_CSV}"

meta = pd.read_csv(META_CSV)

required_cols = {"crop_file_path", "crop_file"}
missing_cols = required_cols - set(meta.columns)
assert not missing_cols, f"metadata.csv is missing columns: {missing_cols}"

In [10]:
meta

,original_image,image_path,box_num,image_id,crop_file,crop_file_path,x_min,y_min,x_max,y_max
0,obs_10317797_photo_14298032_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,1,obs_10317797_photo_14298032_box1,obs_10317797_photo_14298032_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,0,557,218,688
1,obs_10317797_photo_14298032_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,2,obs_10317797_photo_14298032_box1,obs_10317797_photo_14298032_box1_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,2,1,652,722
2,obs_10317797_photo_14298032_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,1,obs_10317797_photo_14298032_box2,obs_10317797_photo_14298032_box2_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,1,4,231,132
3,obs_10317797_photo_14298032_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,2,obs_10317797_photo_14298032_box2,obs_10317797_photo_14298032_box2_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,33,24,84,66
4,obs_130480768_photo_70327211_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,1,obs_130480768_photo_70327211_box1,obs_130480768_photo_70327211_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,11,12,2005,810
...,...,...,...,...,...,...,...,...,...,...
313,obs_340848514_photo_620354575_box4.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,3,obs_340848514_photo_620354575_box4,obs_340848514_photo_620354575_box4_box3.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,84,215,130,262
314,obs_340848514_photo_620354575_box4.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,4,obs_340848514_photo_620354575_box4,obs_340848514_photo_620354575_box4_box4.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,7,4,216,471
315,obs_340848514_photo_620354575_box4.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,5,obs_340848514_photo_620354575_box4,obs_340848514_photo_620354575_box4_box5.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,82,214,131,365
316,obs_64497720_photo_103704678_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,1,obs_64497720_photo_103704678_box1,obs_64497720_photo_103704678_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,11,9,756,406


In [11]:
# #converts paths into absolute paths.
# def resolve_path(p):
#     if pd.isna(p):
#         return None
#     p = str(p).strip()
#     if os.path.isabs(p):
#         return p
#     return str((PROJECT_DIR / p).resolve())

# meta["masked_crop_path"] = meta["masked_crop_path"].apply(resolve_path)

In [11]:
meta = meta[meta["crop_file_path"].notna()].copy() #remove rows where crop path is missing
meta["crop_exists"] = meta["crop_file_path"].apply(os.path.exists)

missing_count = (~meta["crop_exists"]).sum()
if missing_count > 0:
    print(f"Skipping {missing_count} rows because crop file does not exist.")

meta = meta[meta["crop_exists"]].copy()  #only rows where the crop file actually exists.



In [12]:
keep_cols = ["crop_file", "crop_file_path", "image_id"]
if "box_num" in meta.columns:
    keep_cols.append("box_num")

crop_df = meta[keep_cols].drop_duplicates(subset=["crop_file_path"]).reset_index(drop=True) #creates a clean crop table without deduplicates

print("Total valid cropped images:", len(crop_df))
crop_df.head()

Total valid cropped images: 318


,crop_file,crop_file_path,image_id,box_num
0,obs_10317797_photo_14298032_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box1,1
1,obs_10317797_photo_14298032_box1_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box1,2
2,obs_10317797_photo_14298032_box2_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box2,1
3,obs_10317797_photo_14298032_box2_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box2,2
4,obs_130480768_photo_70327211_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_130480768_photo_70327211_box1,1


# Run species classification on all cropped images

In [14]:
# test = classifier.predict(
#     images="/gpfs_common/share03/ftrscape/snair3/wew_notebooks/outputs_crop/obs_250956763_photo_448935283_box2_copy.JPG",
#     rank=Rank.SPECIES, #Predict at species level.
#     k=TOP_K, #Return the top 5 predictions for each image.
#     batch_size=BATCH_SIZE, #Process 1 image at a time.
# )
# test_df=pd.DataFrame(test)

# # print("Total prediction rows:", len(pred_df))
# test_df.head()

In [13]:
crop_paths = crop_df["crop_file_path"].tolist() #Collect all crop image paths into a list.

predictions = classifier.predict(
    images=crop_paths, #Pass the cropped image files to the model.
    rank=Rank.SPECIES, #Predict at species level.
    k=TOP_K, #Return the top 5 predictions for each image.
    batch_size=BATCH_SIZE, #Process 1 image at a time.
)

pred_df = pd.DataFrame(predictions) #Turns the prediction output into a table.

print("Total prediction rows:", len(pred_df))
pred_df.head()

100%|██████████| 318/318 [00:18<00:00, 17.45images/s]


Total prediction rows: 1590


,file_name,kingdom,phylum,class,order,family,genus,species_epithet,species,common_name,score
0,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Odontasteridae,Odontaster,penicillatus,Odontaster penicillatus,,0.060668
1,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Goniasteridae,Ceramaster,japonicus,Ceramaster japonicus,Bat star,0.060482
2,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Goniasteridae,Stellaster,princeps,Stellaster princeps,,0.055059
3,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Asteropseidae,Dermasterias,imbricata,Dermasterias imbricata,,0.044492
4,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Paxillosida,Astropectinidae,Dipsacaster,pretiosus,Dipsacaster pretiosus,,0.042861


# Convert top-k predictions into one row per cropped image

In [14]:
assert "file_name" in pred_df.columns, "Expected 'file_name' in prediction output"
assert "score" in pred_df.columns, "Expected 'score' in prediction output"
assert "species" in pred_df.columns, "Expected 'species' in prediction output"

In [15]:
pred_df

,file_name,kingdom,phylum,class,order,family,genus,species_epithet,species,common_name,score
0,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Odontasteridae,Odontaster,penicillatus,Odontaster penicillatus,,0.060668
1,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Goniasteridae,Ceramaster,japonicus,Ceramaster japonicus,Bat star,0.060482
2,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Goniasteridae,Stellaster,princeps,Stellaster princeps,,0.055059
3,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Asteropseidae,Dermasterias,imbricata,Dermasterias imbricata,,0.044492
4,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Paxillosida,Astropectinidae,Dipsacaster,pretiosus,Dipsacaster pretiosus,,0.042861
...,...,...,...,...,...,...,...,...,...,...,...
1585,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,Aves,Anseriformes,Anatidae,Bucephala,albeola,Bucephala albeola,Bufflehead,0.786600
1586,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,,Siluriformes,Ictaluridae,Ameiurus,nebulosus,Ameiurus nebulosus,Brown bullhead,0.006554
1587,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,,Siluriformes,Ictaluridae,Ameiurus,catus,Ameiurus catus,White catfish,0.005098
1588,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,,Anguilliformes,Anguillidae,Anguilla,rostrata,Anguilla rostrata,American eel,0.004807


In [16]:
summary_rows = []

for crop_path, group in pred_df.groupby("file_name", sort=False):
    group = group.sort_values("score", ascending=False).reset_index(drop=True) #This ensures highest-scoring prediction comes first.
    top1 = group.iloc[0]
    top2 = group.iloc[1]
    top3 = group.iloc[2]
    top4 = group.iloc[3]
    top5 = group.iloc[4]

    topk_records = group[["species", "common_name", "score"]].to_dict(orient="records") #keeps all top-k predictions, but only the most useful columns

    #summary row: one final record per crop, best prediction at top level and top-k stored as JSON
    row = {
        "crop_file_path": crop_path,
        "top1_species": top1.get("species"),
        "top2_species": top2.get("species"),
        "top3_species": top3.get("species"),
        "top4_species": top4.get("species"),
        "top5_species": top5.get("species"),
        "top1_common_name": top1.get("common_name"),
        "top1_score": top1.get("score"),
        "topk_predictions_json": json.dumps(topk_records, ensure_ascii=False),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

final_df = crop_df.merge(summary_df, on="crop_file_path", how="left") #why left? keeps all crops from crop_df, even if somehow a prediction row is missing.

# final_df["low_confidence_flag"] = final_df["top1_score"] < 0.20

print("Final rows:", len(final_df))
final_df.head()

Final rows: 318


,crop_file,crop_file_path,image_id,box_num,top1_species,top2_species,top3_species,top4_species,top5_species,top1_common_name,top1_score,topk_predictions_json
0,obs_10317797_photo_14298032_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box1,1,Odontaster penicillatus,Ceramaster japonicus,Stellaster princeps,Dermasterias imbricata,Dipsacaster pretiosus,,0.060668,"[{""species"": ""Odontaster penicillatus"", ""commo..."
1,obs_10317797_photo_14298032_box1_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box1,2,Larus glaucescens,Larus occidentalis,Larus californicus,Larus vegae,Larus smithsonianus,Glaucous-winged gull,0.885151,"[{""species"": ""Larus glaucescens"", ""common_name..."
2,obs_10317797_photo_14298032_box2_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box2,1,Odontaster penicillatus,Dipsacaster pretiosus,Dermasterias imbricata,Patiria pectinifera,Ceramaster japonicus,,0.057790,"[{""species"": ""Odontaster penicillatus"", ""commo..."
3,obs_10317797_photo_14298032_box2_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box2,2,Chlamydotis macqueenii,Termitomyces titanicus,Sphingonotus candidus,Laemonema yarrellii,Termitomyces schimperi,Macqueen's bustard,0.003206,"[{""species"": ""Chlamydotis macqueenii"", ""common..."
4,obs_130480768_photo_70327211_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_130480768_photo_70327211_box1,1,Erpobdella nigricollis,Gordius robustus,Erpobdella testacea,Erpobdella melanostoma,Erpobdella punctata,,0.132697,"[{""species"": ""Erpobdella nigricollis"", ""common..."


In [17]:
summary_rows = []

for crop_path, group in pred_df.groupby("file_name", sort=False):
    group = group.sort_values("score", ascending=False).reset_index(drop=True) #This ensures highest-scoring prediction comes first.
    top1 = group.iloc[0]
    top2 = group.iloc[1]
    top3 = group.iloc[2]
    top4 = group.iloc[3]
    top5 = group.iloc[4]

    topk_records = group[["species", "common_name", "score"]].to_dict(orient="records") #keeps all top-k predictions, but only the most useful columns

    #summary row: one final record per crop, best prediction at top level and top-k stored as JSON
    for i in range(5):
        no=i+1
        no=str(no)
        val="top"+no+"_species"
        sc="top"+no+"_score"
        cc="top"+no+"_common_name"
        row = {
            "crop_file_path": crop_path,
            # val:group.iloc[i].get("species"),
            "species": group.iloc[i].get("species"),
            # "top2_species": top2.get("species"),
            # "top3_species": top3.get("species"),
            # "top4_species": top4.get("species"),
            # "top5_species": top5.get("species"),
            "common_name": group.iloc[i].get("common_name"),
            "kingdom": group.iloc[i].get("kingdom"),
            "phylum": group.iloc[i].get("phylum"),
            "class": group.iloc[i].get("class"),
            "order": group.iloc[i].get("order"),
            "family": group.iloc[i].get("family"),
            "genus": group.iloc[i].get("genus"),
            "score": group.iloc[i].get("score"),
            "top_no":no,
            "topk_predictions_json": json.dumps(topk_records, ensure_ascii=False),
        }
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

full_df = crop_df.merge(summary_df, on="crop_file_path", how="left") #why left? keeps all crops from crop_df, even if somehow a prediction row is missing.

# final_df["low_confidence_flag"] = final_df["top1_score"] < 0.20

print("Final rows:", len(full_df))
full_df.head()

Final rows: 1590


,crop_file,crop_file_path,image_id,box_num,species,common_name,kingdom,phylum,class,order,family,genus,score,top_no,topk_predictions_json
0,obs_10317797_photo_14298032_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box1,1,Odontaster penicillatus,,Animalia,Echinodermata,Asteroidea,Valvatida,Odontasteridae,Odontaster,0.060668,1,"[{""species"": ""Odontaster penicillatus"", ""commo..."
1,obs_10317797_photo_14298032_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box1,1,Ceramaster japonicus,Bat star,Animalia,Echinodermata,Asteroidea,Valvatida,Goniasteridae,Ceramaster,0.060482,2,"[{""species"": ""Odontaster penicillatus"", ""commo..."
2,obs_10317797_photo_14298032_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box1,1,Stellaster princeps,,Animalia,Echinodermata,Asteroidea,Valvatida,Goniasteridae,Stellaster,0.055059,3,"[{""species"": ""Odontaster penicillatus"", ""commo..."
3,obs_10317797_photo_14298032_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box1,1,Dermasterias imbricata,,Animalia,Echinodermata,Asteroidea,Valvatida,Asteropseidae,Dermasterias,0.044492,4,"[{""species"": ""Odontaster penicillatus"", ""commo..."
4,obs_10317797_photo_14298032_box1_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10317797_photo_14298032_box1,1,Dipsacaster pretiosus,,Animalia,Echinodermata,Asteroidea,Paxillosida,Astropectinidae,Dipsacaster,0.042861,5,"[{""species"": ""Odontaster penicillatus"", ""commo..."


# Save final output CSV

In [ ]:
OUT_CSV = OUTPUTS_BCCP_DIR / "bioclip_species_predictions.csv"
final_df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Saved: /rs1/researchers/a/amallav/results/outputs_bioclip_crop/bioclip_species_predictions_2.csv


In [ ]:
OUT_CSV = OUTPUTS_BCCP_DIR / "results_crop.csv"
full_df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Saved: /rs1/researchers/a/amallav/results/outputs_bioclip_crop/results_crop_2.csv
